# 09 Reranking (Hybrid Candidates, Document-Unknown)

This notebook reranks the **document-unknown hybrid retrieval candidates** using a cross-encoder model.

## Objective
- Load hybrid retrieval outputs
- Group candidate chunks per query
- Score query-chunk pairs with a reranker
- Produce final reranked top-k results
- Save reranked outputs for downstream QA/evaluation

In [16]:
import json
from pathlib import Path

import pandas as pd
from sentence_transformers import CrossEncoder
from tqdm import tqdm

In [17]:
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"

RERANK_TOP_N = 20
FINAL_TOP_K = 10

USE_QUERY_LIMIT = False
QUERY_LIMIT = 50

rerank_config = {
    "retrieval_type": "hybrid_reranked",
    "rerank_model": RERANK_MODEL,
    "rerank_top_n": RERANK_TOP_N,
    "final_top_k": FINAL_TOP_K,
    "document_known": False,
    "candidate_source": "hybrid"
}

rerank_config

{'retrieval_type': 'hybrid_reranked',
 'rerank_model': 'BAAI/bge-reranker-v2-m3',
 'rerank_top_n': 20,
 'final_top_k': 10,
 'document_known': False,
 'candidate_source': 'hybrid'}

In [18]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RETRIEVAL_DIR = PROCESSED_DIR / "retrieval_results"

HYBRID_RESULTS_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid.csv"
HYBRID_RESULTS_PARQUET_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid.parquet"

RERANK_RESULTS_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid_reranked.csv"
RERANK_RESULTS_PARQUET_PATH = RETRIEVAL_DIR / "retrieval_results_hybrid_reranked.parquet"
RERANK_MANIFEST_PATH = RETRIEVAL_DIR / "retrieval_manifest_hybrid_reranked.csv"
RERANK_STATS_PATH = RETRIEVAL_DIR / "retrieval_stats_hybrid_reranked.json"

print("RETRIEVAL_DIR:", RETRIEVAL_DIR)

RETRIEVAL_DIR: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\retrieval_results


In [19]:
if HYBRID_RESULTS_PARQUET_PATH.exists():
    hybrid_df = pd.read_parquet(HYBRID_RESULTS_PARQUET_PATH)
elif HYBRID_RESULTS_CSV_PATH.exists():
    hybrid_df = pd.read_csv(HYBRID_RESULTS_CSV_PATH)
else:
    raise FileNotFoundError("Hybrid retrieval results not found.")

print("hybrid_df shape:", hybrid_df.shape)
print(hybrid_df.columns.tolist())
hybrid_df.head(2)

hybrid_df shape: (1500, 21)
['query_row', 'financebench_id', 'question', 'question_clean', 'expanded_question', 'expected_doc_name', 'expected_company', 'retrieved_rank', 'rrf_score', 'dense_rank', 'dense_score', 'bm25_rank', 'bm25_score', 'embedding_row_idx', 'chunk_id', 'retrieved_doc_id', 'chunk_index', 'chunk_text', 'char_count', 'token_estimate', 'doc_match']


,query_row,financebench_id,question,question_clean,expanded_question,expected_doc_name,expected_company,retrieved_rank,rrf_score,dense_rank,...,bm25_rank,bm25_score,embedding_row_idx,chunk_id,retrieved_doc_id,chunk_index,chunk_text,char_count,token_estimate,doc_match
0,0,financebench_id_03029,What is the FY2018 capital expenditure amount ...,What is the FY2018 capital expenditure amount ...,what is the fy2018 capital expenditure amount ...,3M_2018_10K,3M,1,0.037311,2,...,3.0,37.351639,170,3M_2018_10K_chunk_0172,3M_2018_10K,172,| Years ended December 31 (Millions) | 2018 | ...,786,196,True
1,0,financebench_id_03029,What is the FY2018 capital expenditure amount ...,What is the FY2018 capital expenditure amount ...,what is the fy2018 capital expenditure amount ...,3M_2018_10K,3M,2,0.036755,3,...,1.0,39.849178,675,3M_2022_10K_chunk_0140,3M_2022_10K,140,Refer to the preceding 'Cash Flows from Operat...,1152,288,False


In [20]:
required_cols = [
    "financebench_id",
    "question",
    "question_clean",
    "expanded_question",
    "retrieved_rank",
    "chunk_id",
    "retrieved_doc_id",
    "chunk_text",
]

missing_cols = [c for c in required_cols if c not in hybrid_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in hybrid_df: {missing_cols}")

print("Hybrid retrieval columns OK.")

Hybrid retrieval columns OK.


In [21]:
if USE_QUERY_LIMIT:
    keep_ids = hybrid_df["financebench_id"].drop_duplicates().head(QUERY_LIMIT).tolist()
    hybrid_df = hybrid_df[hybrid_df["financebench_id"].isin(keep_ids)].copy().reset_index(drop=True)

print("hybrid_df shape after optional limit:", hybrid_df.shape)
print("n unique queries:", hybrid_df["financebench_id"].nunique())

hybrid_df shape after optional limit: (1500, 21)
n unique queries: 150


In [22]:
reranker = CrossEncoder(RERANK_MODEL)
print("Loaded reranker:", RERANK_MODEL)

Loaded reranker: BAAI/bge-reranker-v2-m3


In [23]:
def rerank_candidates(question: str, candidate_df: pd.DataFrame, top_n: int = RERANK_TOP_N):
    candidate_df = candidate_df.sort_values("retrieved_rank", ascending=True).head(top_n).copy()

    pairs = [(question, chunk_text) for chunk_text in candidate_df["chunk_text"].tolist()]
    scores = reranker.predict(pairs)

    candidate_df["rerank_score"] = scores
    candidate_df = candidate_df.sort_values("rerank_score", ascending=False).reset_index(drop=True)

    return candidate_df

In [24]:
reranked_records = []
manifest_records = []

grouped = hybrid_df.groupby("financebench_id", sort=False)

for financebench_id, group in tqdm(grouped, total=hybrid_df["financebench_id"].nunique(), desc="Running reranking"):
    first_row = group.iloc[0]

    question = first_row["question"]
    question_clean = first_row["question_clean"]
    expanded_question = first_row["expanded_question"]
    expected_doc_name = first_row.get("expected_doc_name")
    expected_company = first_row.get("expected_company")

    manifest_record = {
        "financebench_id": financebench_id,
        "question": question,
        "expected_doc_name": expected_doc_name,
        "status": None,
        "error_message": None,
        "n_input_candidates": int(len(group)),
        "n_reranked_results": 0
    }

    try:
        reranked_group = rerank_candidates(
            question=question_clean,
            candidate_df=group,
            top_n=min(RERANK_TOP_N, len(group))
        )

        reranked_group = reranked_group.head(FINAL_TOP_K).copy().reset_index(drop=True)

        for new_rank, (_, row) in enumerate(reranked_group.iterrows(), start=1):
            reranked_records.append({
                "financebench_id": financebench_id,
                "question": question,
                "question_clean": question_clean,
                "expanded_question": expanded_question,
                "expected_doc_name": expected_doc_name,
                "expected_company": expected_company,
                "retrieved_rank": new_rank,
                "rerank_score": float(row["rerank_score"]),
                "previous_rank": row["retrieved_rank"],
                "rrf_score": row.get("rrf_score"),
                "dense_rank": row.get("dense_rank"),
                "dense_score": row.get("dense_score"),
                "bm25_rank": row.get("bm25_rank"),
                "bm25_score": row.get("bm25_score"),
                "chunk_id": row["chunk_id"],
                "retrieved_doc_id": row["retrieved_doc_id"],
                "chunk_index": row.get("chunk_index"),
                "chunk_text": row["chunk_text"],
                "char_count": row.get("char_count"),
                "token_estimate": row.get("token_estimate"),
            })

        manifest_record["status"] = "success"
        manifest_record["n_reranked_results"] = int(len(reranked_group))

    except Exception as e:
        manifest_record["status"] = "error"
        manifest_record["error_message"] = str(e)

    manifest_records.append(manifest_record)

reranked_df = pd.DataFrame(reranked_records)
rerank_manifest_df = pd.DataFrame(manifest_records)

print("reranked_df shape:", reranked_df.shape)
print("rerank_manifest_df shape:", rerank_manifest_df.shape)

Running reranking: 100%|██████████| 150/150 [27:59<00:00, 11.20s/it]

reranked_df shape: (1500, 20)
rerank_manifest_df shape: (150, 7)


In [25]:
if "expected_doc_name" not in reranked_df.columns:
    reranked_df["expected_doc_name"] = None

reranked_df["doc_match"] = (
    reranked_df["expected_doc_name"].fillna("").astype(str)
    == reranked_df["retrieved_doc_id"].fillna("").astype(str)
)

reranked_df[[
    "financebench_id",
    "retrieved_rank",
    "previous_rank",
    "retrieved_doc_id",
    "expected_doc_name",
    "doc_match",
    "rerank_score"
]].head(15)

,financebench_id,retrieved_rank,previous_rank,retrieved_doc_id,expected_doc_name,doc_match,rerank_score
0,financebench_id_03029,1,5,3M_2018_10K,3M_2018_10K,True,0.916973
1,financebench_id_03029,2,9,COCACOLA_2017_10K,3M_2018_10K,False,0.893710
2,financebench_id_03029,3,1,3M_2018_10K,3M_2018_10K,True,0.810469
3,financebench_id_03029,4,2,3M_2022_10K,3M_2018_10K,False,0.542732
4,financebench_id_03029,5,3,3M_2023Q2_10Q,3M_2018_10K,False,0.466677
5,financebench_id_03029,6,8,3M_2018_10K,3M_2018_10K,True,0.323688
6,financebench_id_03029,7,10,LOCKHEEDMARTIN_2022_10K,3M_2018_10K,False,0.233005
7,financebench_id_03029,8,6,MGMRESORTS_2018_10K,3M_2018_10K,False,0.179325
8,financebench_id_03029,9,7,ADOBE_2017_10K,3M_2018_10K,False,0.139304
9,financebench_id_03029,10,4,WALMART_2018_10K,3M_2018_10K,False,0.105698


In [26]:
top1_df = reranked_df[reranked_df["retrieved_rank"] == 1].copy()

top1_doc_match_rate = float(top1_df["doc_match"].mean()) if len(top1_df) else 0.0
topk_doc_match_rate = float(
    reranked_df.groupby("financebench_id")["doc_match"].max().mean()
) if len(reranked_df) else 0.0

summary_df = pd.DataFrame([{
    "n_queries": int(reranked_df["financebench_id"].nunique()),
    "top1_doc_match_rate": top1_doc_match_rate,
    f"top{FINAL_TOP_K}_doc_match_rate": topk_doc_match_rate
}])

summary_df

,n_queries,top1_doc_match_rate,top10_doc_match_rate
0,150,0.633333,0.886667


In [27]:
reranked_df.to_csv(RERANK_RESULTS_CSV_PATH, index=False, encoding="utf-8")
reranked_df.to_parquet(RERANK_RESULTS_PARQUET_PATH, index=False)

rerank_manifest_df.to_csv(RERANK_MANIFEST_PATH, index=False, encoding="utf-8")

print("Saved:")
print("-", RERANK_RESULTS_CSV_PATH)
print("-", RERANK_RESULTS_PARQUET_PATH)
print("-", RERANK_MANIFEST_PATH)

Saved:
- C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\retrieval_results\retrieval_results_hybrid_reranked.csv
- C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\retrieval_results\retrieval_results_hybrid_reranked.parquet
- C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\retrieval_results\retrieval_manifest_hybrid_reranked.csv


In [28]:
rerank_stats = {
    "retrieval_type": "hybrid_reranked",
    "document_known": False,
    "rerank_model": RERANK_MODEL,
    "rerank_top_n": RERANK_TOP_N,
    "final_top_k": FINAL_TOP_K,
    "n_queries": int(reranked_df["financebench_id"].nunique()),
    "n_result_rows": int(len(reranked_df)),
    "n_manifest_rows": int(len(rerank_manifest_df)),
    "top1_doc_match_rate": top1_doc_match_rate,
    f"top{FINAL_TOP_K}_doc_match_rate": topk_doc_match_rate,
    "results_csv": str(RERANK_RESULTS_CSV_PATH),
    "results_parquet": str(RERANK_RESULTS_PARQUET_PATH),
    "manifest_csv": str(RERANK_MANIFEST_PATH),
}

with open(RERANK_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(rerank_stats, f, indent=2, ensure_ascii=False)

print("Saved stats:", RERANK_STATS_PATH)
rerank_stats

Saved stats: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\processed\retrieval_results\retrieval_stats_hybrid_reranked.json


{'retrieval_type': 'hybrid_reranked',
 'document_known': False,
 'rerank_model': 'BAAI/bge-reranker-v2-m3',
 'rerank_top_n': 20,
 'final_top_k': 10,
 'n_queries': 150,
 'n_result_rows': 1500,
 'n_manifest_rows': 150,
 'top1_doc_match_rate': 0.6333333333333333,
 'top10_doc_match_rate': 0.8866666666666667,
 'results_csv': 'C:\\Users\\ntheo\\Desktop\\Repositories\\Financial-RAG\\thesis-rag\\data\\processed\\retrieval_results\\retrieval_results_hybrid_reranked.csv',
 'results_parquet': 'C:\\Users\\ntheo\\Desktop\\Repositories\\Financial-RAG\\thesis-rag\\data\\processed\\retrieval_results\\retrieval_results_hybrid_reranked.parquet',
 'manifest_csv': 'C:\\Users\\ntheo\\Desktop\\Repositories\\Financial-RAG\\thesis-rag\\data\\processed\\retrieval_results\\retrieval_manifest_hybrid_reranked.csv'}

In [29]:
print(rerank_manifest_df["status"].value_counts(dropna=False))
rerank_manifest_df[["financebench_id", "status", "error_message"]].head(10)

status
success    150
Name: count, dtype: int64


,financebench_id,status,error_message
0,financebench_id_03029,success,None
1,financebench_id_04672,success,None
2,financebench_id_00499,success,None
3,financebench_id_01226,success,None
4,financebench_id_01865,success,None
5,financebench_id_00807,success,None
6,financebench_id_00941,success,None
7,financebench_id_01858,success,None
8,financebench_id_02987,success,None
9,financebench_id_07966,success,None


In [30]:
reranked_df[[
    "financebench_id",
    "question",
    "retrieved_rank",
    "previous_rank",
    "retrieved_doc_id",
    "expected_doc_name",
    "doc_match",
    "rerank_score",
    "chunk_id"
]].head(20)

,financebench_id,question,retrieved_rank,previous_rank,retrieved_doc_id,expected_doc_name,doc_match,rerank_score,chunk_id
0,financebench_id_03029,What is the FY2018 capital expenditure amount ...,1,5,3M_2018_10K,3M_2018_10K,True,0.916973,3M_2018_10K_chunk_0290
1,financebench_id_03029,What is the FY2018 capital expenditure amount ...,2,9,COCACOLA_2017_10K,3M_2018_10K,False,0.893710,COCACOLA_2017_10K_chunk_0288
2,financebench_id_03029,What is the FY2018 capital expenditure amount ...,3,1,3M_2018_10K,3M_2018_10K,True,0.810469,3M_2018_10K_chunk_0172
3,financebench_id_03029,What is the FY2018 capital expenditure amount ...,4,2,3M_2022_10K,3M_2018_10K,False,0.542732,3M_2022_10K_chunk_0140
4,financebench_id_03029,What is the FY2018 capital expenditure amount ...,5,3,3M_2023Q2_10Q,3M_2018_10K,False,0.466677,3M_2023Q2_10Q_chunk_0274
5,financebench_id_03029,What is the FY2018 capital expenditure amount ...,6,8,3M_2018_10K,3M_2018_10K,True,0.323688,3M_2018_10K_chunk_0157
6,financebench_id_03029,What is the FY2018 capital expenditure amount ...,7,10,LOCKHEEDMARTIN_2022_10K,3M_2018_10K,False,0.233005,LOCKHEEDMARTIN_2022_10K_chunk_0204
7,financebench_id_03029,What is the FY2018 capital expenditure amount ...,8,6,MGMRESORTS_2018_10K,3M_2018_10K,False,0.179325,MGMRESORTS_2018_10K_chunk_0142
8,financebench_id_03029,What is the FY2018 capital expenditure amount ...,9,7,ADOBE_2017_10K,3M_2018_10K,False,0.139304,ADOBE_2017_10K_chunk_0199
9,financebench_id_03029,What is the FY2018 capital expenditure amount ...,10,4,WALMART_2018_10K,3M_2018_10K,False,0.105698,WALMART_2018_10K_chunk_0182


## Συμπέρασμα

Σε αυτό το notebook:

- φορτώσαμε τα hybrid retrieval candidates
- εφαρμόσαμε cross-encoder reranking
- κρατήσαμε τα τελικά reranked top-k results
- αποθηκεύσαμε results, manifest και stats

Το επόμενο notebook θα χρησιμοποιήσει dense / hybrid / hybrid-reranked retrieval outputs για το QA stage.